In [ ]:
#Lets Start with the imports
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report
import os
import numpy as np
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)


In [ ]:
# Task 1: Write your code here:
# read the CSV file useing
cvs = os.path.join(path , 'Q1_data.csv')
df = pd.read_csv(cvs)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Delivery_Time  distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()
# A bit Skiwed but its ok tbh I'll try later to fix it

In [ ]:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# TWO handels one for the others and one for the target
# for the other features
print("\nMissing Values (df.isnull().sum()):")
print(df.isnull().sum())
# Ill fill them with the The most coomen for all the object coulms
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
  df[col]=df[col].fillna(df[col].mode()[0])

# for the float I'll get the meanian
df['Courier_Experience_yrs']= df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())
# Target has to be removed the row
df= df.dropna(subset=['Delivery_Time'])
df['Delivery_Time']

In [ ]:
# Task 3: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()
if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
df.head()

In [ ]:
df['Delivery_Time'].isnull().sum()


In [ ]:
df.drop('Delivery_Time', axis= 1)

In [ ]:
# Task 4: Write your code here:
# For this i will use the above Catigorical i got

categorical_cols = df.select_dtypes(include=["object"]).columns

encoder = OneHotEncoder(sparse_output=False)
df[categorical_cols]= encoder.fit_transform(df[categorical_cols]) # I dont know whats the problem asked tA Also did not now

In [ ]:
le =LabelEncoder()
for col in categorical_cols:
  df[col]=le.fit_transform(df[col]) # dont know why i have problems in the one hot its the same

In [ ]:
# Task 5: Write your code here:
standard_scaler = StandardScaler() # Instantiate StandardScaler
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

df[numerical_cols]= standard_scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 6: Write your code here: #TBH no need ican see if form the disturbution and i see it normaly
# if you want to be a perfectinost use the log transformation might make a better model
# but no time to see BYEEEEE


In [ ]:
from sklearn.model_selection import KFold , train_test_split , StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
X= df.drop('Delivery_Time' , axis=1)
y = df['Delivery_Time']
# i did it above

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold , train_test_split , StratifiedKFold
# Even though the data is not complitly skewed i will still use straified KFolds
n_splits = 3 # K=3 Folds

kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
# Storage for linear regression results for each fold
lr_losses = []
lr_mse = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)
  # Validate
  y_pred = model.predict(X_test)

  # Calculate evaluation metrics
  mse = sklearn_mse(y_test, y_pred)
  rmse = np.sqrt(mse)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_mse.append(mse)

print("Linear Regression Results")
print(f"  Average MSE: {np.mean(lr_mse):.4f}")

In [ ]:
importances = {}

importances['Random Forest'] = model.feature_importances_
#importances['XGBoost'] = sklearn_models['XGBoost'].feature_importances_
#importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show() # the most important is the distance

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('predicted Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: